# XGBoost — Purged Temporal Backtest

This notebook compares core behavioral features with the extended subscription-aware feature set using the same purged out-of-time split as Logistic Regression.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from xgboost import XGBClassifier

DATA_PATH = Path("../data/churn_modeling_ready.csv")
MODEL_DIR = Path("../models")
OUTPUT_DIR = Path("../data")
MODEL_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", None)


In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["observation_date"])

print("Shape:", df.shape)
print("Date range:", df["observation_date"].min(), "to", df["observation_date"].max())
print("Churn observations:", int(df["is_churn"].sum()))
print("Overall churn rate:", f"{df['is_churn'].mean():.2%}")


In [ ]:
core_categorical = ["channel", "country"]

core_numeric = [
    "customer_age",
    "account_tenure_days",
    "post_per_month",
    "newfriend_per_month",
    "like_per_month",
    "adview_per_month",
    "dislike_per_month",
    "unfriend_per_month",
    "message_per_month",
    "reply_per_month",
    "total_engagement",
    "negative_engagement",
    "total_activity",
    "dislike_rate",
    "unfriend_rate",
    "reply_rate",
    "negative_activity_share",
    "adview_share",
]

core_features = core_categorical + core_numeric

extended_categorical = ["channel", "country", "current_plan"]

extended_numeric = core_numeric + [
    "current_mrr",
    "bill_period_months",
    "discount",
    "mrr_change",
    "upgrade_flag",
    "downsell_flag",
    "upgrade_last_90d",
    "downsell_last_90d",
]

extended_features = extended_categorical + extended_numeric

print("Core predictors:", len(core_features))
print("Extended predictors:", len(extended_features))


## Purged out-of-time split

The churn target looks forward approximately one month. The one-month gap prevents training labels from overlapping the test period.


In [ ]:
TEST_START = pd.Timestamp("2020-04-11")
TRAIN_END = TEST_START - pd.DateOffset(months=1)

train_df = df[df["observation_date"] < TRAIN_END].copy()
embargo_df = df[
    (df["observation_date"] >= TRAIN_END)
    & (df["observation_date"] < TEST_START)
].copy()
test_df = df[df["observation_date"] >= TEST_START].copy()

y_train = train_df["is_churn"].astype(int)
y_test = test_df["is_churn"].astype(int)

print("TRAIN")
print("Rows:", len(train_df))
print("Dates:", train_df["observation_date"].min(), "to", train_df["observation_date"].max())
print("Churns:", int(y_train.sum()), "| rate:", f"{y_train.mean():.2%}")

print("\nPURGE / EMBARGO")
print("Rows:", len(embargo_df))
print("Dates:", embargo_df["observation_date"].min(), "to", embargo_df["observation_date"].max())

print("\nTEST")
print("Rows:", len(test_df))
print("Dates:", test_df["observation_date"].min(), "to", test_df["observation_date"].max())
print("Churns:", int(y_test.sum()), "| rate:", f"{y_test.mean():.2%}")


In [ ]:
def make_preprocessor(categorical, numeric):
    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                categorical,
            ),
            ("numeric", "passthrough", numeric),
        ]
    )


core_preprocessor = make_preprocessor(core_categorical, core_numeric)
extended_preprocessor = make_preprocessor(extended_categorical, extended_numeric)

X_train_core = core_preprocessor.fit_transform(train_df[core_features])
X_test_core = core_preprocessor.transform(test_df[core_features])

X_train_extended = extended_preprocessor.fit_transform(train_df[extended_features])
X_test_extended = extended_preprocessor.transform(test_df[extended_features])

print("Core matrices:", X_train_core.shape, X_test_core.shape)
print("Extended matrices:", X_train_extended.shape, X_test_extended.shape)


In [ ]:
xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "n_estimators": 400,
    "learning_rate": 0.03,
    "max_depth": 4,
    "min_child_weight": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
}

core_xgb = XGBClassifier(**xgb_params)
extended_xgb = XGBClassifier(**xgb_params)

core_xgb.fit(X_train_core, y_train)
extended_xgb.fit(X_train_extended, y_train)

core_probability = core_xgb.predict_proba(X_test_core)[:, 1]
extended_probability = extended_xgb.predict_proba(X_test_extended)[:, 1]


In [ ]:
def evaluate_model(name, y_true, probabilities):
    return {
        "model": name,
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities),
    }


core_results = evaluate_model(
    "Core Behavior XGBoost",
    y_test,
    core_probability,
)

extended_results = evaluate_model(
    "Extended Subscription XGBoost",
    y_test,
    extended_probability,
)

xgb_comparison = pd.DataFrame([core_results, extended_results])
display(xgb_comparison)
print("Random PR baseline:", round(y_test.mean(), 4))


## Compare all four models


In [ ]:
logistic_results = pd.read_csv(
    OUTPUT_DIR / "logistic_regression_model_comparison.csv"
)

logistic_core = logistic_results.iloc[0]
logistic_extended = logistic_results.iloc[1]

all_models = pd.DataFrame({
    "model": [
        "Logistic - Core",
        "Logistic - Extended",
        "XGBoost - Core",
        "XGBoost - Extended",
    ],
    "roc_auc": [
        logistic_core["roc_auc"],
        logistic_extended["roc_auc"],
        core_results["roc_auc"],
        extended_results["roc_auc"],
    ],
    "pr_auc": [
        logistic_core["pr_auc"],
        logistic_extended["pr_auc"],
        core_results["pr_auc"],
        extended_results["pr_auc"],
    ],
})

display(all_models.sort_values("pr_auc", ascending=False))


In [ ]:
core_fpr, core_tpr, _ = roc_curve(y_test, core_probability)
ext_fpr, ext_tpr, _ = roc_curve(y_test, extended_probability)

plt.figure(figsize=(7, 5))
plt.plot(core_fpr, core_tpr, label=f"Core (AUC={core_results['roc_auc']:.3f})")
plt.plot(ext_fpr, ext_tpr, label=f"Extended (AUC={extended_results['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("XGBoost ROC Curve")
plt.legend()
plt.tight_layout()
plt.show()

core_precision, core_recall, _ = precision_recall_curve(y_test, core_probability)
ext_precision, ext_recall, _ = precision_recall_curve(y_test, extended_probability)

plt.figure(figsize=(7, 5))
plt.plot(core_recall, core_precision, label=f"Core (AP={core_results['pr_auc']:.3f})")
plt.plot(ext_recall, ext_precision, label=f"Extended (AP={extended_results['pr_auc']:.3f})")
plt.axhline(y=y_test.mean(), linestyle="--", label="Random baseline")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("XGBoost Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.show()


## Top-k lift


In [ ]:
def top_k_metrics(y_true, probabilities, percentages=(0.05, 0.10, 0.20)):
    result = pd.DataFrame({
        "actual": np.asarray(y_true),
        "probability": probabilities,
    }).sort_values("probability", ascending=False)

    overall_rate = result["actual"].mean()
    total_churns = result["actual"].sum()
    rows = []

    for pct in percentages:
        n = max(1, int(np.ceil(len(result) * pct)))
        top = result.head(n)
        top_churns = top["actual"].sum()
        precision = top["actual"].mean()

        rows.append({
            "top_population_pct": pct * 100,
            "customers_targeted": n,
            "churners_found": int(top_churns),
            "precision": precision,
            "recall_of_all_churners": top_churns / total_churns,
            "lift": precision / overall_rate,
        })

    return pd.DataFrame(rows)


extended_lift = top_k_metrics(y_test, extended_probability)
display(extended_lift)


## Feature importance


In [ ]:
feature_names = extended_preprocessor.get_feature_names_out()

importance_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": extended_xgb.feature_importances_,
    })
    .sort_values("importance", ascending=False)
)

display(importance_df.head(25))


## Save comparison models and selected deployment model

The stronger XGBoost variant is selected by PR-AUC on the purged out-of-time test set. Both variants are saved for reproducibility; the Streamlit app loads the generic `xgboost_selected.pkl` artifact.


In [ ]:
if core_results["pr_auc"] >= extended_results["pr_auc"]:
    selected_xgb_name = "XGBoost - Core"
    selected_preprocessor = core_preprocessor
    selected_xgb = core_xgb
    selected_features = core_features
    selected_probability = core_probability
else:
    selected_xgb_name = "XGBoost - Extended"
    selected_preprocessor = extended_preprocessor
    selected_xgb = extended_xgb
    selected_features = extended_features
    selected_probability = extended_probability

print("Selected deployment model:", selected_xgb_name)

predictions = test_df[
    [
        "account_id",
        "observation_date",
        "is_churn",
        "current_plan",
        "current_mrr",
        "upgrade_last_90d",
        "downsell_last_90d",
    ]
].copy()

predictions["xgb_churn_probability"] = selected_probability
predictions["risk_rank"] = predictions["xgb_churn_probability"].rank(
    method="first",
    ascending=False,
)
predictions["monthly_revenue_at_risk"] = (
    predictions["xgb_churn_probability"] * predictions["current_mrr"]
)

predictions = predictions.sort_values(
    "xgb_churn_probability",
    ascending=False,
)

joblib.dump(
    {
        "preprocessor": core_preprocessor,
        "model": core_xgb,
        "features": core_features,
    },
    MODEL_DIR / "xgboost_core.pkl",
)

joblib.dump(
    {
        "preprocessor": extended_preprocessor,
        "model": extended_xgb,
        "features": extended_features,
    },
    MODEL_DIR / "xgboost_extended.pkl",
)

joblib.dump(
    {
        "preprocessor": selected_preprocessor,
        "model": selected_xgb,
        "features": selected_features,
        "model_name": selected_xgb_name,
    },
    MODEL_DIR / "xgboost_selected.pkl",
)

predictions.to_csv(
    OUTPUT_DIR / "xgboost_test_predictions.csv",
    index=False,
)

all_models.to_csv(
    OUTPUT_DIR / "final_model_comparison.csv",
    index=False,
)

print("Saved both XGBoost variants, selected deployment model, predictions, and comparison.")
